In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


In [2]:
# Load the dataset
url = "lexical.csv"
df = pd.read_csv(url)

# Extract features and labels
features = df.iloc[:, 1:-1].values  # Exclude the URL and the target variable

yList = []
for i in range(features.shape[0]):
    yList.append(np.random.randint(0,2))

labels = pd.DataFrame(yList)

labels = labels.iloc[:].values
labels = labels.reshape(-1,1)

# Standardize features
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)


In [3]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_size):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.linear(x)
        out = self.sigmoid(out)
        return out

input_size = X_train.shape[1]
model = LogisticRegressionModel(input_size)


In [4]:
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


In [5]:
print(X_train.shape)
print(y_train.shape)

X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)

X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)


(59698, 15)
(59698, 1)


In [6]:
num_epochs = 1000

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor.view(-1, 1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [100/1000], Loss: 0.7168
Epoch [200/1000], Loss: 0.7104
Epoch [300/1000], Loss: 0.7060
Epoch [400/1000], Loss: 0.7028
Epoch [500/1000], Loss: 0.7003
Epoch [600/1000], Loss: 0.6984
Epoch [700/1000], Loss: 0.6969
Epoch [800/1000], Loss: 0.6958
Epoch [900/1000], Loss: 0.6950
Epoch [1000/1000], Loss: 0.6945


In [7]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    predictions = (predictions > 0.5).float()

accuracy = (predictions == y_test_tensor.view(-1, 1)).sum().item() / len(y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')


Test Accuracy: 50.38%


In [8]:
# Export the model
torch.onnx.export(model,               # model being run
                  X_train_tensor,                         # model input (or a tuple for multiple inputs)
                  "model.onnx",   # where to save the model (can be a file or file-like object)
                  export_params=True,        # store the trained parameter weights inside the model file
                  #opset_version=10,          # the ONNX version to export the model to
                  #do_constant_folding=True,  # whether to execute constant folding for optimization
                  input_names = ['input'],   # the model's input names
                  output_names = ['output'], # the model's output names
                  dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                                'output' : {0 : 'batch_size'}})

In [9]:
model.eval()

# Assuming 'X_test_tensor' is your input tensor
with torch.no_grad():
    output = model(X_test_tensor)


tensor([0.4968])
tensor([[0.],
        [0.],
        [0.],
        ...,
        [0.],
        [0.],
        [0.]])
